# Task 1 — Central Pipeline Notebook

This notebook runs the **complete Task 1 pipeline** end-to-end:

| Step | What happens |
|------|-------------|
| 0 | Load configuration from `config/config.json` |
| 1 | *(Optional)* Convert a PDF → Markdown using Ollama (`pdf_to_md`) |
| 2 | Run the heading-extraction pipeline on the Markdown file |

Run cells top-to-bottom.  
Toggle `RUN_PDF_CONVERSION` in **Cell 2** if you already have a Markdown file and want to skip Step 1.

---
## 0 · Setup — paths & configuration

In [130]:
import os, sys, json

# Make Task1 root importable from anywhere the notebook is launched
TASK1_ROOT = os.path.dirname(os.path.abspath("__file__"))
if TASK1_ROOT not in sys.path:
    sys.path.insert(0, TASK1_ROOT)

# ── Load config.json ──────────────────────────────────────────────────────────
CONFIG_PATH = os.path.join(TASK1_ROOT, "config", "config.json")
with open(CONFIG_PATH, "r", encoding="utf-8") as _f:
    cfg = json.load(_f)

# Resolve paths relative to Task1 root
PDF_SOURCE   = cfg.get("pdf_source", "")
MARKDOWN_PATH = os.path.normpath(os.path.join(TASK1_ROOT, cfg["output"]["markdown_path"]))
OUTPUT_FILE   = os.path.normpath(os.path.join(TASK1_ROOT, cfg["output"]["output_file"]))
OLLAMA_MODEL  = cfg["model"]["model_id"]

if PDF_SOURCE and not os.path.isabs(PDF_SOURCE):
    PDF_SOURCE = os.path.normpath(os.path.join(TASK1_ROOT, PDF_SOURCE))

print("Config loaded successfully")
print(f"  PDF source    : {PDF_SOURCE}")
print(f"  Markdown path : {MARKDOWN_PATH}")
print(f"  Output file   : {OUTPUT_FILE}")
print(f"  Model         : {OLLAMA_MODEL}")

Config loaded successfully
  PDF source    : c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\data\ancestral_heading_docs\GlobalTradeMgmtSWIs_CreatinganOfflineShipmentinGTMv1_1.pdf
  Markdown path : c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\data\Markdowns\GlobalTradeMgmtSWIs_CreatinganOfflineShipmentinGTMv1_1.md
  Output file   : c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\output\md_json_outputs\GlobalTradeMgmtSWIs_CreatinganOfflineShipmentinGTMv1_1.json
  Model         : gemma2:9b


In [131]:
import truststore
truststore.inject_into_ssl()

---
## 1 · (Optional) PDF → Markdown conversion

Set `RUN_PDF_CONVERSION = True` to convert the PDF defined in `config.json → pdf_source`.  
Skip this step if you already have a Markdown file at `config.json → output.markdown_path`.

In [132]:
RUN_PDF_CONVERSION = True   # ← set to True to convert the PDF first

if RUN_PDF_CONVERSION:
    from utils.pdf_to_md import convert_pdf_to_markdown

    if not PDF_SOURCE or not os.path.isfile(PDF_SOURCE):
        raise FileNotFoundError(
            f"PDF not found at '{PDF_SOURCE}'. "
            "Check pdf_source in config/config.json."
        )

    convert_pdf_to_markdown(
        pdf_path=PDF_SOURCE,
        output_md_path=MARKDOWN_PATH,
        ollama_model=OLLAMA_MODEL,
        ollama_base_url=cfg.get("ollama_base_url", "http://localhost:11434"),
    )
else:
    print("PDF conversion skipped.")
    if os.path.isfile(MARKDOWN_PATH):
        print(f"  ✔ Markdown file found at: {MARKDOWN_PATH}")
    else:
        print(f"  ⚠ WARNING: Markdown file does NOT exist at: {MARKDOWN_PATH}")
        print("  → Set RUN_PDF_CONVERSION = True above to generate it from the PDF.")

Starting conversion for: c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\data\ancestral_heading_docs\GlobalTradeMgmtSWIs_CreatinganOfflineShipmentinGTMv1_1.pdf
Using device: cuda


Recognizing Text: 100%|██████████| 8/8 [01:55<00:00, 14.38s/it]
Detecting bboxes: 0it [00:00, ?it/s]

Successfully converted PDF to markdown and saved as c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\data\Markdowns\GlobalTradeMgmtSWIs_CreatinganOfflineShipmentinGTMv1_1.md


---
## 2 · Heading extraction pipeline

In [133]:
import importlib, sys, os
# Force-reload main and config so code/config changes take effect without a kernel restart
for _mod in list(sys.modules.keys()):
    if _mod in ('main', 'config') or _mod.startswith('config.'):
        del sys.modules[_mod]

import main as _main_module
importlib.reload(_main_module)
from main import process_markdown

if not os.path.isfile(MARKDOWN_PATH):
    raise FileNotFoundError(
        f"Markdown file not found at '{MARKDOWN_PATH}'.\n"
        "Run Step 1 (set RUN_PDF_CONVERSION = True) or place a .md file there."
    )

metrics = process_markdown(MARKDOWN_PATH)
print("\nPipeline complete!")


  MARKDOWN HEADING EXTRACTION PIPELINE
  Model: gemma2:9b
  Input: c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\data\Markdowns\GlobalTradeMgmtSWIs_CreatinganOfflineShipmentinGTMv1_1.md

[Step 1] Reading Markdown text
--------------------------------------------------
  Document length: 3,781 characters
  Doc retrieval time: 0.00s

[Step 2] Splitting text into chunks
--------------------------------------------------
  Chunks created: 2

[Step 3] Extracting headings with LangExtract
--------------------------------------------------


LangExtract: Processing, current=2,399 chars, processed=0 chars:  [06:28]
LangExtract: Processing, current=2,399 chars, processed=0 chars:  [06:19]
LangExtract: Processing, current=2,399 chars, processed=0 chars:  [06:26]


  Chunk  1/2: LLM failed - Content must contain an 'extractions' key.
  Chunk  1/2:  3 headings [REGEX fallback]


LangExtract: Processing, current=1,380 chars, processed=0 chars:  [02:09]

    Rejected (not standalone): 'Shipped Quantity'
    Rejected (not standalone): 'UOM'
    Rejected (not standalone): 'Save'
    Rejected (not standalone): 'Finish'
    Rejected (not standalone): 'Submit Offline Shipment'
    Rejected (not standalone): 'Search'
    Rejected (not standalone): 'Indicator'
    Rejected (not standalone): 'Offline Shipment'
    Rejected (not standalone): 'Documents'
    Rejected (not standalone): 'Generate Document'
  Chunk  2/2:  0 headings [LLM] in 129.03s

  ⚠  1 chunk(s) had LLM extraction failures: [1]
     Set 'use_fallback_regex': true in config.json to recover headings from these chunks.
  Total LLM time: 129.03s across 1 calls

[Step 4] Assembling per-chunk LangExtract outputs
--------------------------------------------------
  Chunks returned: 2
  Saved to: c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\output\md_json_outputs\GlobalTradeMgmtSWIs_CreatinganOfflineShipmentinGTMv1_1.json

                    EXTRACTION

---
## 3 · Inspect results

In [134]:
import json, os
# Re-read OUTPUT_FILE from config so it matches what the pipeline just saved
_cfg_path = os.path.join(TASK1_ROOT, "config", "config.json")
with open(_cfg_path, "r", encoding="utf-8") as _f:
    _cfg = json.load(_f)
OUTPUT_FILE = os.path.normpath(os.path.join(TASK1_ROOT, _cfg["output"]["output_file"]))
print(f"Reading: {OUTPUT_FILE}")

with open(OUTPUT_FILE, "r", encoding="utf-8") as _f:
    results = json.load(_f)


print(f"Total chunks in output: {len(results)}")
print("\n--- First chunk preview ---")
first = results[0] if results else {}
print(json.dumps({k: v for k, v in first.items() if k != "Text"}, indent=2, ensure_ascii=False))

Reading: c:\Users\AK8963\OneDrive - Zebra Technologies\Documents\Langextract_run\Task1\output\md_json_outputs\GlobalTradeMgmtSWIs_CreatinganOfflineShipmentinGTMv1_1.json
Total chunks in output: 2

--- First chunk preview ---
{
  "chunk_id1": {
    "Sub Heading 1": "Creating an Offline Shipment in GTM .....",
    "Sub Heading 2": "Carrier name .....",
    "Sub Heading 3": "12. Involved Parties (Ship From, Bill To, and Ship To) ....."
  },
  "Metadata": "\"Sub heading\": \"Creating an Offline Shipment in GTM: ........Version 1.1 - 1) Navigate yourself to the GTM website[: https://otmgtm-a577711.otm.us2.oraclecloud.com](https://otmgtm-a577711.otm.us2.oraclecloud.com/) -If asked for a password please use your Zebra'............\", \"Sub heading\": \"Carrier name: ........- 2. **Transport Mode** Select from list - 3. **INCO Term** Select from list - 4. **INCO Term Location** City Name based on INCO term - 5. **Reference Number** Tracking number - 6. **Ship To Contact**............\", \"Sub 